In [ ]:
import cv2
import os
import subprocess
from ultralytics import YOLO

# Load models
VOD = "path-to-your-video.mp4"
classify_model = YOLO("runs/classify/train/weights/best.pt")
killfeed_box_model = YOLO("killfeed/results/run21/weights/best.pt")

# Killfeed region
killfeed_x, killfeed_y, killfeed_w, killfeed_h = 850, 50, 430, 350

# Save images to this folder
save_dir = "killfeed_frames"
os.makedirs(save_dir, exist_ok=True)
image_id = 0

# Open video
cap = cv2.VideoCapture(VOD)
batch_size = 10
frame_buffer = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_buffer.append(frame)

    if len(frame_buffer) == batch_size:
        for f in frame_buffer:
            results = classify_model(f)
            if results[0].probs.top1 == 1:
                detection = killfeed_box_model(f)

                for result in detection:
                    for box in result.boxes:
                        x_min, y_min, x_max, y_max = map(int, box.xyxy[0])

                        # Ensure bounding box is within or near the killfeed region
                        buffer = 5
                        if (
                            x_min >= killfeed_x - buffer and x_max <= killfeed_x + killfeed_w + buffer and
                            y_min >= killfeed_y - buffer and y_max <= killfeed_y + killfeed_h + buffer
                        ):
                            crop = f[y_min:y_max, x_min:x_max]
                            if crop.shape[0] > 0 and crop.shape[1] > 0:
                                filename = os.path.join(save_dir, f"kill_{image_id:05d}.jpg")
                                cv2.imwrite(filename, crop)
                                image_id += 1
        frame_buffer.clear()

cap.release()
cv2.destroyAllWindows()

print(f"✅ Saved {image_id} killfeed crops to: {save_dir}")

# Launch labelImg on the folder
try:
    subprocess.run(["labelImg", save_dir], check=True)
except FileNotFoundError:
    print("⚠️ labelImg not found. Make sure it's installed and added to PATH.")
